# 

In [1]:
import Pkg
Pkg.activate(".")

  Activating project at `~/Work/jlProjects/QEDsandbox/GPUEventGenerators.jl`


In [2]:
using Revise

In [3]:
using Random
using KernelAbstractions
using GPUEventGenerators

RNG = Xoshiro(137)

[ Info: Precompiling GPUEventGenerators [90f62166-f53e-5083-ac0e-6e2bf74182ae] (cache misses: mismatched flags (2), include_dependency fhash change (2))


Xoshiro(0x3d49b847083ba6c7, 0xb24070c1aecda5eb, 0x0df056308b5b0cd2, 0x1210222cfda290b3, 0x182da8b9bcddcd87)

In [4]:
Pkg.add("Metal")
using Metal

   Resolving package versions...
      Compat entries added for 
  No Changes to `~/Work/jlProjects/QEDsandbox/GPUEventGenerators.jl/Project.toml`
  No Changes to `~/Work/jlProjects/QEDsandbox/GPUEventGenerators.jl/Manifest.toml`


In [5]:
Metal.functional()

true

In [6]:
Metal.versioninfo()

macOS 14.6.1, Darwin 23.6.0

Toolchain:
- Julia: 1.11.3
- LLVM: 16.0.6

Julia packages: 
- Metal.jl: 1.5.1
- GPUArrays: 11.2.2
- GPUCompiler: 1.3.2
- KernelAbstractions: 0.9.34
- ObjectiveC: 3.4.1
- LLVM: 9.2.0
- LLVMDowngrader_jll: 0.6.0+0

1 device:
- Apple M1 Pro (64.000 KiB allocated)


In [7]:
# Utility

@inline Base.zero(::Type{Tuple{}}) = ()
@inline Base.zero(::Type{Tuple{Vararg{T, N}}}) where {T, N} = (zero(T), zero(NTuple{N - 1, T})...)


In [39]:
N = 256
VECTOR_T = MtlVector
T = Float32
PAYLOAD_T = NTuple{4, T}   # make a "bigger" payload type

# we don't really care for the test what the payload is
payload = VECTOR_T(rand(RNG, PAYLOAD_T, N))
@show typeof(payload)
weights = VECTOR_T(rand(RNG, T, N)) # calculated "probability"
random_numbers = VECTOR_T(rand(RNG, T, N)) # random values to filter against

@show eltype(payload)

out_payload = VECTOR_T(zeros(eltype(payload), size(payload)))   # output buffer 1
out_weights = VECTOR_T(zeros(eltype(weights), size(weights)))   # output buffer 2
accepted_count = VECTOR_T(zeros(Int32, 1)) # global memory counter

BACKEND = get_backend(payload)


typeof(payload) = MtlVector{NTuple{4, Float32}, Metal.PrivateStorage}
eltype(payload) = NTuple{4, Float32}


MetalBackend()

In [40]:
filter_scan(BACKEND, 32)(payload, weights, random_numbers, out_payload, out_weights, accepted_count; ndrange = N)
KernelAbstractions.synchronize(BACKEND)